In [1]:
import torch
import torchvision

print("🟢 PyTorch version:", torch.__version__)
print("🟢 Torchvision version:", torchvision.__version__)
print("🟢 CUDA available:", torch.cuda.is_available())
print("🟢 CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

try:
    import detectron2
    print("🟢 Detectron2 module is importable.")
except ImportError:
    print("🔴 Detectron2 is NOT installed.")


🟢 PyTorch version: 2.6.0+cu124
🟢 Torchvision version: 0.21.0+cu124
🟢 CUDA available: True
🟢 CUDA device: Tesla T4
🔴 Detectron2 is NOT installed.


In [ ]:
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities (e.g. compiled operators).
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

# Properly install detectron2. (Please do not install twice in both ways)
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [3]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
torch:  2.6 ; cuda:  cu124
detectron2: 0.6


In [4]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

In [5]:
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog

# Register datasets
register_coco_instances(
    "train", {}, 
    "/kaggle/input/multimodal-coco-ann/train.json", 
    "/kaggle/input/multimodal/images/train"
)

import json
from pathlib import Path

# Read original val.json
val_json_path = "/kaggle/input/multimodal-coco-ann/val.json"
with open(val_json_path, "r") as f:
    data = json.load(f)

# Add dummy 'info' if missing
if "info" not in data:
    data["info"] = {
        "description": "Autogenerated for Detectron2 evaluation",
        "version": "1.0",
        "year": 2025
    }

# Save fixed version to working directory
fixed_val_path = "/kaggle/working/val_fixed.json"
with open(fixed_val_path, "w") as f:
    json.dump(data, f)

# Re-register using the patched file
from detectron2.data.datasets import register_coco_instances

register_coco_instances(
    "multimodal_val_fixed", {},
    fixed_val_path,
    "/kaggle/input/multimodal/images/val"
)


# Optional: test set if needed later
register_coco_instances(
    "test", {}, 
    "/kaggle/input/multimodal-coco-ann/test.json", 
    "/kaggle/input/multimodal/images/test"
)

# (Optional) check metadata
MetadataCatalog.get("train").thing_classes = [
    "credit_card", "passport", "drivers_license", 
    "student_id", "mail", "receipt", "ticket"
]


In [7]:
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.evaluation import COCOEvaluator
import os
import shutil

# === Setup config ===
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/retinanet_R_101_FPN_3x.yaml"))

# ✅ Dataset
cfg.DATASETS.TRAIN = ("train",)
cfg.DATASETS.TEST = ("multimodal_val_fixed",)
cfg.DATALOADER.SAMPLER_TRAIN = "RepeatFactorTrainingSampler"
cfg.DATALOADER.REPEAT_THRESHOLD = 0.1
cfg.DATALOADER.NUM_WORKERS = 2

# ✅ RetinaNet-specific
cfg.MODEL.RETINANET.NUM_CLASSES = 7
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = 0.05  # low for evaluation

# ✅ Training
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.0001
cfg.SOLVER.MAX_ITER = 90000
cfg.SOLVER.CHECKPOINT_PERIOD = 10000
cfg.SOLVER.STEPS = []  # no decay

# ✅ Output + Resume
cfg.OUTPUT_DIR = "./output_retinanet"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# === Handle checkpoint copy from input ===
checkpoint_dir = "/kaggle/input/multimodal-retinanet"
if os.path.exists(os.path.join(checkpoint_dir, "model_0049999.pth")):
    # Copy the checkpoint
    shutil.copy(os.path.join(checkpoint_dir, "model_0049999.pth"),
                os.path.join(cfg.OUTPUT_DIR, "model_0049999.pth"))

    # Manually set last_checkpoint to match the file
    with open(os.path.join(cfg.OUTPUT_DIR, "last_checkpoint"), "w") as f:
        f.write("model_0049999.pth\n")

    cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_0049999.pth")
else:
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/retinanet_R_101_FPN_3x.yaml")

# ✅ Custom trainer with COCOEvaluator that includes mAP@0.50 extraction
class MyTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, cfg, True, output_folder)

trainer = MyTrainer(cfg)
trainer.resume_or_load(resume=True)
trainer.train()


[08/03 06:22:25 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

/usr/local/lib/python3.11/dist-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[08/03 06:22:44 d2.utils.events]:  eta: 7:10:10  iter: 50019  total_loss: 0.01791  loss_cls: 0.002097  loss_box_reg: 0.01576    time: 0.6477  last_time: 0.4235  data_time: 0.0607  last_data_time: 0.0035   lr: 0.0001  max_mem: 3183M


2025-08-03 06:22:47.683472: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754202168.009413      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754202168.098624      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


[08/03 06:23:13 d2.utils.events]:  eta: 7:04:07  iter: 50039  total_loss: 0.0224  loss_cls: 0.001884  loss_box_reg: 0.01883    time: 0.6281  last_time: 0.5668  data_time: 0.0171  last_data_time: 0.0072   lr: 0.0001  max_mem: 3183M
[08/03 06:23:25 d2.utils.events]:  eta: 7:02:23  iter: 50059  total_loss: 0.02565  loss_cls: 0.003699  loss_box_reg: 0.02112    time: 0.6245  last_time: 0.5869  data_time: 0.0054  last_data_time: 0.0053   lr: 0.0001  max_mem: 3243M
[08/03 06:23:39 d2.utils.events]:  eta: 7:03:41  iter: 50079  total_loss: 0.01405  loss_cls: 0.0013  loss_box_reg: 0.01286    time: 0.6416  last_time: 0.5762  data_time: 0.0360  last_data_time: 0.0040   lr: 0.0001  max_mem: 3243M
[08/03 06:23:53 d2.utils.events]:  eta: 7:05:37  iter: 50099  total_loss: 0.01858  loss_cls: 0.002155  loss_box_reg: 0.01626    time: 0.6503  last_time: 0.7394  data_time: 0.0233  last_data_time: 0.0042   lr: 0.0001  max_mem: 3243M
[08/03 06:24:07 d2.utils.events]:  eta: 7:07:37  iter: 50119  total_loss: 0

In [ ]:
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
import os
import shutil

cfg = get_cfg()

# ✅ Load Cascade Mask R-CNN config
cfg.merge_from_file(model_zoo.get_config_file("Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = ""  # Do NOT load pretrained weights if resuming
# ✅ Dataset settings
cfg.DATASETS.TRAIN = ("train",)
cfg.DATASETS.TEST = ("multimodal_val_fixed",)
cfg.DATALOADER.SAMPLER_TRAIN = "RepeatFactorTrainingSampler"
cfg.DATALOADER.REPEAT_THRESHOLD = 0.1
cfg.DATALOADER.NUM_WORKERS = 2

# ✅ Model settings
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 7
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.MASK_ON = False

# ✅ Training hyperparameters
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.0001
cfg.SOLVER.MAX_ITER = 70000
cfg.SOLVER.CHECKPOINT_PERIOD = 10000
cfg.SOLVER.STEPS = []

# ✅ Output dir
cfg.OUTPUT_DIR = "./output_cascade_rcnn"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ✅ Copy weights and checkpoint files from input dir
pretrained_dir = "/kaggle/input/multimodal-cascade"
shutil.copy(os.path.join(pretrained_dir, "model_0039999.pth"), cfg.OUTPUT_DIR)
shutil.copy(os.path.join(pretrained_dir, "last_checkpoint"), cfg.OUTPUT_DIR)

# ✅ Resume from checkpoint
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_0039999.pth")

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=True)
trainer.train()


In [ ]:
# Copy model weights
shutil.copy("/kaggle/input/multimodal-cascade/model_0039999.pth", cfg.OUTPUT_DIR)

# Manually write correct last_checkpoint file
with open(os.path.join(cfg.OUTPUT_DIR, "last_checkpoint"), "w") as f:
    f.write("model_0039999.pth\n")

# Set MODEL.WEIGHTS to empty — will use last_checkpoint automatically
cfg.MODEL.WEIGHTS = ""

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=True)
trainer.train()


In [ ]:
"""
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
import os
import shutil 

cfg = get_cfg()

# Load pretrained config
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml"))
#cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml")

# Datasets
cfg.DATASETS.TRAIN = ("train",)
cfg.DATASETS.TEST = ("multimodal_val_fixed",)
cfg.DATALOADER.SAMPLER_TRAIN = "RepeatFactorTrainingSampler"
cfg.DATALOADER.REPEAT_THRESHOLD = 0.1  # default is 0.0; adjust as needed
cfg.DATALOADER.NUM_WORKERS = 2

# Output dir
cfg.OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

shutil.copy("/kaggle/input/multimodal-model3/last_checkpoint", cfg.OUTPUT_DIR)
shutil.copy("/kaggle/input/multimodal-model3/model_0029999.pth", cfg.OUTPUT_DIR)
# Overwrite last_checkpoint manually
with open(os.path.join(cfg.OUTPUT_DIR, "last_checkpoint"), "w") as f:
    f.write("model_0029999.pth")

# Model settings
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 7  # your 7 classes
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # threshold for inference

# Training hyperparams
cfg.SOLVER.IMS_PER_BATCH = 2  # fits in Kaggle GPU
cfg.SOLVER.BASE_LR = 0.0001  # learning rate
cfg.SOLVER.MAX_ITER = 40000  # total iterations (~3-5 epochs depending on data size)
cfg.SOLVER.CHECKPOINT_PERIOD = 5000 
cfg.SOLVER.STEPS = []  # no LR decay

# Trainer
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=True)
trainer.train()
"""


In [17]:
import json
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from collections import defaultdict

# === Paths ===
gt_path = "/kaggle/working/val_fixed.json"
pred_path = "/kaggle/working/output_retinanet/inference/coco_instances_results.json"

# === Load GT and predictions ===
coco_gt = COCO(gt_path)
coco_gt.dataset.setdefault("info", {})
coco_gt.createIndex()

with open(pred_path, "r") as f:
    preds = json.load(f)
coco_dt = coco_gt.loadRes(preds)

# === Evaluate ===
coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
coco_eval.evaluate()
coco_eval.accumulate()

# === Prep class info ===
precision = coco_eval.eval["precision"]
cat_ids = coco_eval.params.catIds
cat_id_to_name = {cat["id"]: cat["name"] for cat in coco_gt.loadCats(cat_ids)}

# === Count GT instances and images with annotations ===
images_per_class = defaultdict(set)
instances_per_class = defaultdict(int)
gt_boxes = defaultdict(lambda: defaultdict(list))

for ann in coco_gt.dataset["annotations"]:
    cid = ann["category_id"]
    img_id = ann["image_id"]
    images_per_class[cid].add(img_id)
    instances_per_class[cid] += 1
    gt_boxes[img_id][cid].append({"bbox": ann["bbox"], "matched": False})

# === Match predictions to GT for TP/FP ===
IOU_THRESH = 0.5
tp_fp_counts = defaultdict(lambda: {"TP": 0, "FP": 0, "FN": 0})

def iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    xa = max(x1, x2)
    ya = max(y1, y2)
    xb = min(x1 + w1, x2 + w2)
    yb = min(y1 + h1, y2 + h2)
    inter = max(0, xb - xa) * max(0, yb - ya)
    union = w1 * h1 + w2 * h2 - inter
    return inter / union if union > 0 else 0

for pred in preds:
    img_id = pred["image_id"]
    pred_box = pred["bbox"]
    pred_cat = pred["category_id"]
    matched = False

    for gt in gt_boxes[img_id].get(pred_cat, []):
        if not gt["matched"] and iou(pred_box, gt["bbox"]) >= IOU_THRESH:
            gt["matched"] = True
            tp_fp_counts[pred_cat]["TP"] += 1
            matched = True
            break

    if not matched:
        tp_fp_counts[pred_cat]["FP"] += 1

# === Count FN
for img_id in gt_boxes:
    for cid in gt_boxes[img_id]:
        for gt in gt_boxes[img_id][cid]:
            if not gt["matched"]:
                tp_fp_counts[cid]["FN"] += 1

# === Format output ===
print("YOLO")
print(f"{'':>20} {'images':>10} {'GT':>10} {'Prec':>10} {'Recall':>10} {'mAP@.5':>10} {'mAP@.5:.95':>10}")

# === Overall stats ===
total_TP = sum(tp_fp_counts[c]["TP"] for c in cat_ids)
total_FP = sum(tp_fp_counts[c]["FP"] for c in cat_ids)
total_FN = sum(tp_fp_counts[c]["FN"] for c in cat_ids)
total_imgs = len(set(coco_gt.getImgIds()))
total_gt = sum(instances_per_class.values())

prec = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
rec = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
map50 = np.mean(precision[0, :, :, 0, 0][precision[0, :, :, 0, 0] > -1])
map5095 = np.mean(precision[:, :, :, 0, 0][precision[:, :, :, 0, 0] > -1])
print(f"{'all':>20} {total_imgs:>10} {total_gt:>10} {prec:>10.3f} {rec:>10.3f} {map50:>10.3f} {map5095:>10.3f}")

# === Per-class stats ===
for idx, cid in enumerate(cat_ids):
    name = cat_id_to_name[cid]
    TP = tp_fp_counts[cid]["TP"]
    FP = tp_fp_counts[cid]["FP"]
    FN = tp_fp_counts[cid]["FN"]
    img_count = len(images_per_class[cid])
    inst_count = instances_per_class[cid]

    prec = TP / (TP + FP) if (TP + FP) > 0 else 0
    rec = TP / (TP + FN) if (TP + FN) > 0 else 0

    p50 = precision[0, :, idx, 0, 0]
    p5095 = precision[:, :, idx, 0, 0]
    ap50 = np.mean(p50[p50 > -1]) if np.any(p50 > -1) else 0.0
    ap5095 = np.mean(p5095[p5095 > -1]) if np.any(p5095 > -1) else 0.0

    print(f"{name:>20} {img_count:>10} {inst_count:>10} {prec:>10.3f} {rec:>10.3f} {ap50:>10.3f} {ap5095:>10.3f}")


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.24s).
Accumulating evaluation results...
DONE (t=0.09s).
YOLO
                         images         GT       Prec     Recall     mAP@.5 mAP@.5:.95
                 all        505        284      0.086      0.771      0.449      0.428
         credit_card         12         59      0.101      0.746      0.131      0.111
            passport         23         23      0.065      0.957      0.848      0.841
     drivers_license          9         13      0.086      0.769      0.458      0.441
          student_id          9         14      0.022      0.643      0.295      0.295
                mail         21         23      0.038      0.957      0.462      0.444
             receipt         32         46      0.163     

In [17]:
import os
import json
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# === Step 1: Load ground truth and predictions ===
gt_path = "/kaggle/input/multimodal-coco-ann/val.json"
pred_path = "/kaggle/working/output_retinanet/coco_instances_results.json"

coco_gt = COCO(gt_path)
coco_gt.dataset.setdefault("info", {})  # Patch if missing
coco_gt.createIndex()

with open(pred_path, "r") as f:
    coco_preds = json.load(f)

coco_dt = coco_gt.loadRes(coco_preds)

# === Step 2: Run COCO Evaluation ===
coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
coco_eval.evaluate()
coco_eval.accumulate()

# === Step 3: Extract correct per-class AP@0.50 ===
precision = coco_eval.eval['precision']  # shape [10, 101, K, 1, 1]
iou_index = 0  # for IoU = 0.50
cat_ids = coco_eval.params.catIds  # category IDs in the same order as eval
cat_id_to_name = {cat['id']: cat['name'] for cat in coco_gt.loadCats(cat_ids)}

# === Sanity Check: Print category mapping ===
print("== Category ID to Name Mapping ==")
for i, cid in enumerate(cat_ids):
    print(f"{i}: ID={cid} → {cat_id_to_name[cid]}")
print()

# === Step 4: Print per-class AP@0.50 ===
print("Per-class AP@0.50:")
print(f"{'Class':<20} {'AP@0.50':>8}")
print("-" * 30)

for class_idx, cat_id in enumerate(cat_ids):
    class_name = cat_id_to_name[cat_id]
    p = precision[iou_index, :, class_idx, 0, 0]  # IoU=0.50
    valid = p[p > -1]
    ap50 = np.mean(valid) if valid.size > 0 else float('nan')
    print(f"{class_name:<20} {ap50 * 100:>8.2f}")


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.13s).
Accumulating evaluation results...
DONE (t=0.07s).
== Category ID to Name Mapping ==
0: ID=0 → credit_card
1: ID=1 → passport
2: ID=2 → drivers_license
3: ID=3 → student_id
4: ID=4 → mail
5: ID=5 → receipt
6: ID=6 → ticket

Per-class AP@0.50:
Class                 AP@0.50
------------------------------
credit_card             12.16
passport                81.51
drivers_license         49.38
student_id              29.06
mail                    43.21
receipt                 44.81
ticket                  45.14
